# 🎨 Cartoonify — Unified Interface

Combines **Reimagine** (FLUX.1-Kontext-dev), **Scene** (Depth ControlNet), and **Portrait**
(Canny ControlNet) into a single interface. One story. One button. Three rendering styles.

**Flow:** Write your story → upload a photo → pick a rendering style → click Cartoonify.
Gemini builds the structured prompt silently on generate — no separate Build Prompt step.

**Settings modes:**
- **Default** — hardcoded parameters per rendering style
- **Wild** — Gemini reads the story and suggests parameters optimised for maximum satirical impact

| Style | Pipeline | Best for |
|---|---|---|
| **Reimagine** | FLUX.1-Kontext-dev | Creative reinterpretation — scene can change freely |
| **Scene** | FLUX.1-dev + ControlNet Depth | Crowds, architecture — spatial layout preserved |
| **Portrait** | FLUX.1-dev + ControlNet Canny | Specific person must be immediately recognisable |

**VRAM note:** Only one pipeline loads at a time. Switching modes takes ~60–90 s (one-time per switch).

**Requires:** Google Colab Pro → Runtime → Change runtime type → **A100 GPU**

> Accept the [FLUX.1-Kontext-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev)
> on HuggingFace before running — it is gated separately from FLUX.1-dev.

---

## 1 — Confirm A100 GPU

In [ ]:
!nvidia-smi

## 2 — Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece
!pip install --quiet gradio
!pip install --quiet huggingface_hub
!pip install --quiet google-genai
!pip install --quiet opencv-python-headless

In [ ]:
import os
os.kill(os.getpid(), 9)
print("IGNORE: session crashed. This is intentional — continue to the next cell.")

## 3 — Imports

In [ ]:
import gc
import json
import queue
import threading
import datetime
import os
import numpy as np
import cv2
import torch
import gradio as gr
import google.genai as genai
import google.genai.types as genai_types
from PIL import Image
from diffusers import (
    FluxKontextPipeline,
    FluxControlNetPipeline,
    FluxControlNetModel,
)
from transformers import pipeline as hf_pipeline

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'cv2    : {cv2.__version__}')
print(f'gradio : {gr.__version__}')
print(f'genai  : {genai.__version__}')

## 4 — Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')

## 5 — API Tokens

**Hugging Face** — FLUX.1-dev and FLUX.1-Kontext-dev are both gated models.
Add a Read token to Colab Secrets as `HF_TOKEN`.

**Google Gemini** — Add your key to Colab Secrets as `GOOGLE_API_KEY`
([aistudio.google.com/apikey](https://aistudio.google.com/apikey)).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN       = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ Logged in to Hugging Face')
print('✓ Google API key loaded' if GOOGLE_API_KEY else '⚠️  GOOGLE_API_KEY not found — story-to-prompt and Wild mode will not work')

## 6 — Configuration

Edit this cell to change the LoRA, trigger word, default rendering mode, or default settings mode.
Nothing else needs to change.

In [ ]:
# ── LoRA ───────────────────────────────────────────────────────────────────────────────
LORA_SOURCE     = 'drive'   # 'huggingface' | 'drive'
LORA_HF_REPO    = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_HF_WEIGHT  = 'Ghibili-Cartoon-Art.safetensors'
LORA_DRIVE_PATH = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors'

# ── Trigger + fallback prompt ────────────────────────────────────────────────────
DEFAULT_TRIGGER = 'gdo_cartoon'
DEFAULT_PROMPT  = 'satirical cartoon illustration, bold outlines, vivid flat colours, expressive exaggerated features'

# ── Gemini ──────────────────────────────────────────────────────────────────────────
GOOGLE_MODEL = 'gemini-2.5-flash-lite'

# ── Base models ────────────────────────────────────────────────────────────────────
BASE_MODEL_KONTEXT = 'black-forest-labs/FLUX.1-Kontext-dev'
BASE_MODEL_FLUX    = 'black-forest-labs/FLUX.1-dev'
CONTROLNET_MODEL   = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'
DEPTH_MODEL        = 'depth-anything/Depth-Anything-V2-Small-hf'

# ── Default parameters per rendering mode ────────────────────────────────────────
DEFAULTS = {
    'reimagine': {'guidance': 2.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'scene':     {'guidance': 3.5, 'cn_scale': 0.8, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'portrait':  {'guidance': 3.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
}
DEFAULT_STEPS         = 28
DEFAULT_SEED          = 42
DEFAULT_MODE          = 'Reimagine'   # Reimagine | Scene | Portrait
DEFAULT_SETTINGS_MODE = 'Default'     # Default | Wild

# ── Output directory ────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Default mode          : {DEFAULT_MODE}')
print(f'Default settings mode : {DEFAULT_SETTINGS_MODE}')
print(f'LoRA source           : {LORA_SOURCE}')
print(f'Output dir            : {OUTPUT_DIR}')

## 7 — Load Models

`load_pipeline(mode)` handles VRAM switching — unloads the current pipeline before loading the new one.
The initial load uses `DEFAULT_MODE` set in cell-config.

⏱️ First run downloads ~24–29 GB depending on mode. Warm cache: ~1 minute.

In [ ]:
active_mode     = None
pipe            = None
controlnet      = None
depth_estimator = None


def _peft_patch():
    import peft.import_utils as _peft_utils
    import peft.tuners.lora.torchao as _peft_torchao
    _orig = _peft_utils.is_torchao_available
    def _safe():
        try: return _orig()
        except ImportError: return False
    _peft_utils.is_torchao_available = _safe
    _peft_torchao.is_torchao_available = _safe


def load_pipeline(mode: str) -> None:
    """Load the pipeline for the given mode, unloading the current one first."""
    global pipe, controlnet, depth_estimator, active_mode

    mode = mode.lower()
    if mode == active_mode:
        print(f'✓ {mode} already loaded.')
        return

    # ── Unload ───────────────────────────────────────────────────────────────────
    if active_mode:
        print(f'Unloading {active_mode} pipeline...')
    if pipe is not None:
        del pipe
        pipe = None
    if controlnet is not None:
        del controlnet
        controlnet = None
    if depth_estimator is not None:
        del depth_estimator
        depth_estimator = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')

    # ── Load ────────────────────────────────────────────────────────────────────
    if mode == 'reimagine':
        print('Loading FLUX.1-Kontext-dev...')
        pipe = FluxKontextPipeline.from_pretrained(
            BASE_MODEL_KONTEXT, torch_dtype=torch.bfloat16
        ).to('cuda')

    elif mode in ('scene', 'portrait'):
        print('Loading ControlNet...')
        controlnet = FluxControlNetModel.from_pretrained(
            CONTROLNET_MODEL, torch_dtype=torch.bfloat16
        )
        print('Loading FLUX.1-dev...')
        pipe = FluxControlNetPipeline.from_pretrained(
            BASE_MODEL_FLUX, controlnet=controlnet, torch_dtype=torch.bfloat16
        ).to('cuda')
        if mode == 'scene':
            print('Loading depth estimator (CPU)...')
            depth_estimator = hf_pipeline(
                'depth-estimation', model=DEPTH_MODEL, device='cpu'
            )
    else:
        raise ValueError(f'Unknown mode: {mode!r}')

    # ── LoRA ──────────────────────────────────────────────────────────────────────
    _peft_patch()
    if LORA_SOURCE == 'huggingface':
        pipe.load_lora_weights(LORA_HF_REPO, weight_name=LORA_HF_WEIGHT)
    else:
        pipe.load_lora_weights(LORA_DRIVE_PATH)

    active_mode = mode
    print(f'✅ {mode} pipeline ready.')


# ── Initial load ──────────────────────────────────────────────────────────────
load_pipeline(DEFAULT_MODE.lower())

## 8 — Story-to-Prompt and Wild Settings (Gemini)

Two Gemini functions:
- `build_prompt_from_story()` — Default mode: returns structured prompt only
- `build_wild_settings()` — Wild mode: returns prompt + parameter suggestions optimised for satirical impact

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEFAULT MODE — prompt only
# ─────────────────────────────────────────────────────────────────────────────

GEMINI_SYSTEM_PROMPT = """You are a prompt engineer for a FLUX.1 LoRA fine-tuned on editorial and political cartoons.
Convert the user's story into a structured image generation prompt.

OUTPUT FORMAT — one line only, no explanation, no preamble:
gdo_cartoon <medium>, <technique>, <color>, <mood>, <commentary>, <composition>

RULES:
- Always start with: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration
- Always include: pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight
- Default color: black and white | monochrome | print cartoon | white background
  (use spot color red only if violence or urgency is central to the story)
- Derive mood, commentary, and composition from the story
- Pipe-separated keywords within each layer; comma between layers
- Max 5 keywords per layer
- Composition: estimate figure count, suggest layout, add speech bubble hint if there is dialogue

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | dark humour | critical | bleak, political commentary | scathing | power critique | deadpan | accusatory, two-group layout | standing figure left | queue of figures right | speech bubble top | eye level | wide shot

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, absurdist | dark | pompous | grotesque | confrontational, political condemnation | war metaphor | accusatory | ironic | scathing, large figure dominates upper frame | tiny figures lower third | desk as stage | top-heavy composition | eye level

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | duplicitous | tense | darkly comic, political commentary | diplomatic hypocrisy | sharp | deadpan | allegorical, two figure layout | shadow duality | frontal handshake foreground | fighting silhouettes background | centred composition | eye level"""


def build_prompt_from_story(story: str, trigger_word: str) -> str:
    if not story.strip():
        raise ValueError('Empty story.')
    if not GOOGLE_API_KEY:
        raise ValueError('GOOGLE_API_KEY not set in Colab Secrets.')

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=GEMINI_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=300,
        ),
    )
    prompt = response.text.strip()

    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'gdo_cartoon':
        prompt = prompt.replace('gdo_cartoon', active_trigger, 1)

    return prompt


# ─────────────────────────────────────────────────────────────────────────────
# WILD MODE — prompt + parameter suggestions for maximum satirical impact
# ─────────────────────────────────────────────────────────────────────────────

WILD_SYSTEM_PROMPT = """You are a prompt engineer AND generation parameter optimizer for a FLUX.1 LoRA fine-tuned on satirical editorial cartoons.
Your goal: craft a structured prompt AND suggest generation parameters that together produce the sharpest, most satirically impactful cartoon possible.

OUTPUT FORMAT — exactly two lines, nothing else:
Line 1 (prompt): gdo_cartoon <medium>, <technique>, <color>, <mood>, <commentary>, <composition>
Line 2 (settings): {"mode": "reimagine|scene|portrait", "guidance": <2.0-6.0>, "steps": <28-40>, "cn_scale": <0.3-1.2>, "cn_end": <0.50-1.0>, "canny_low": <10-100>, "canny_high": <80-400>, "rationale": "<one sentence>"}

PROMPT RULES (same as Default):
- Start: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration
- Always include: pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight
- Default color: black and white | monochrome | print cartoon | white background
- Pipe-separated within layer, comma between layers, max 5 keywords per layer

MODE SELECTION:
- reimagine: total recomposition wanted, identity less critical, surreal/allegorical staging
- scene: power hierarchy, crowd, spatial layout must be preserved
- portrait: specific face must be immediately recognisable — caricature target

PARAMETER RULES FOR SATIRICAL IMPACT:
- Confrontational / scathing / dark → guidance 4.5–5.5, steps 35–40
- Whimsical / absurdist / ironic → guidance 2.5–3.5, steps 28–32
- Face identity critical → portrait, canny_low 20–40, canny_high 100–160, cn_scale 0.85–1.0
- Crowd / hierarchy scene → scene, cn_scale 0.85–0.95, steps 34–38
- Total recomposition / allegory → reimagine, guidance 2.5–3.0
- cn_end: 0.70 for more FLUX finishing freedom (richer hand-drawn feel), 0.80 standard, 0.85+ strict edge lock
- More complex compositions → more steps (36–40); single figure → 28–32 is sufficient

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Line 1: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | dark humour | critical | bleak, political commentary | scathing | power critique | deadpan | accusatory, two-group layout | standing figure left | queue of figures right | speech bubble top | eye level | wide shot
Line 2: {"mode": "scene", "guidance": 4.5, "steps": 36, "cn_scale": 0.85, "cn_end": 0.75, "canny_low": 50, "canny_high": 200, "rationale": "Crowd-heavy power imbalance — depth map locks two-group layout; high guidance drives scathing political tone"}

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Line 1: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, absurdist | dark | pompous | grotesque | confrontational, political condemnation | war metaphor | accusatory | ironic | scathing, large figure dominates upper frame | tiny figures lower third | desk as stage | top-heavy composition | eye level
Line 2: {"mode": "portrait", "guidance": 5.0, "steps": 38, "cn_scale": 0.90, "cn_end": 0.70, "canny_low": 28, "canny_high": 130, "rationale": "General's pomposity demands tight face outlines; high guidance and steps maximise grotesque caricature detail"}

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Line 1: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | duplicitous | tense | darkly comic, political commentary | diplomatic hypocrisy | sharp | deadpan | allegorical, two figure layout | shadow duality | frontal handshake foreground | fighting silhouettes background | centred composition | eye level
Line 2: {"mode": "reimagine", "guidance": 3.0, "steps": 32, "cn_scale": 0.70, "cn_end": 0.80, "canny_low": 50, "canny_high": 200, "rationale": "Shadow duality allegory needs Kontext's semantic recomposition — standard guidance lets FLUX interpret the staging freely"}"""


def build_wild_settings(story: str, trigger_word: str) -> tuple:
    """Returns (prompt_str, settings_dict) for Wild mode.
    settings_dict keys: mode, guidance, steps, cn_scale, cn_end, canny_low, canny_high, rationale.
    Returns (DEFAULT_PROMPT, {}) on any failure.
    """
    if not story.strip() or not GOOGLE_API_KEY:
        return DEFAULT_PROMPT, {}

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=WILD_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=500,
        ),
    )

    lines = [l.strip() for l in response.text.strip().split('\n') if l.strip()]
    prompt = lines[0] if lines else DEFAULT_PROMPT

    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'gdo_cartoon':
        prompt = prompt.replace('gdo_cartoon', active_trigger, 1)

    settings = {}
    for line in lines[1:]:
        if line.startswith('{'):
            try:
                settings = json.loads(line)
            except json.JSONDecodeError:
                pass
            break

    return prompt, settings


print('✓ Default build_prompt_from_story ready')
print('✓ Wild build_wild_settings ready')

## 9 — Inference

In [ ]:

# CSS-animated spinner shown in the result panel while FLUX renders
SPINNER_HTML = (
    '<div style="display:flex;flex-direction:column;align-items:center;justify-content:center;'
    'height:580px;border-radius:12px;'
    'background:linear-gradient(160deg,#faf5ff 0%,#f0ebff 100%);'
    'border:2px dashed #c4b5fd;gap:1.5rem;">'
    '<div style="width:72px;height:72px;'
    'border:7px solid #ede9fe;border-top-color:#7c3aed;border-right-color:#a78bfa;'
    'border-radius:50%;animation:cfy-spin 0.8s linear infinite;"></div>'
    '<p style="color:#7c3aed;font-weight:700;font-size:0.85rem;margin:0;'
    'letter-spacing:0.12em;">RENDERING</p>'
    '<style>@keyframes cfy-spin{to{transform:rotate(360deg);}}</style>'
    '</div>'
)


def extract_canny(pil_image: Image.Image, low: int, high: int) -> Image.Image:
    img_np = np.array(pil_image)
    edges  = cv2.Canny(img_np, low, high)
    edges  = edges[:, :, None]
    edges  = np.concatenate([edges, edges, edges], axis=2)
    return Image.fromarray(edges)


def _progress_bar(step: int, total: int, width: int = 20) -> str:
    filled = int(width * step / max(total, 1))
    bar = '█' * filled + '░' * (width - filled)
    pct = int(100 * step / max(total, 1))
    return f'⧗ FLUX rendering... {bar} {step}/{total} ({pct}%)'


def cartoonify(
    story,
    image,
    mode_label,
    settings_mode,
    trigger_word,
    guidance_scale,
    num_steps,
    cn_scale,
    cn_end,
    canny_low,
    canny_high,
    seed,
    prompt_override,
):
    """Generator. Yields (result_image, log_text, spinner_html, guidance, steps, cn_scale, cn_end, canny_low, canny_high)."""
    if image is None:
        raise gr.Error('Upload a photo first.')

    mode = mode_label.lower()
    log_lines = []

    def log(msg):
        log_lines.append(msg)
        return '\n'.join(log_lines)

    def log_update_last(msg):
        if log_lines:
            log_lines[-1] = msg
        else:
            log_lines.append(msg)
        return '\n'.join(log_lines)

    def emit(image_val, log_text,
             spinner=None,      # None=no change; str=update ('' clears, SPINNER_HTML shows)
             show_result=None,  # None=no change; False=hide result panel; True implied by image_val
             g=None, s=None, cn_s=None, cn_e=None, clow=None, chigh=None):
        # Build the result_output update
        if show_result is False:
            res = gr.update(visible=False)
        elif image_val is not None:
            res = gr.update(value=image_val, visible=True)
        else:
            res = gr.update()

        return (
            res,
            log_text,
            gr.update() if spinner is None else gr.update(value=spinner),
            gr.update() if g     is None else gr.update(value=g),
            gr.update() if s     is None else gr.update(value=s),
            gr.update() if cn_s  is None else gr.update(value=cn_s),
            gr.update() if cn_e  is None else gr.update(value=cn_e),
            gr.update() if clow  is None else gr.update(value=clow),
            gr.update() if chigh is None else gr.update(value=chigh),
        )

    def flux_with_progress(pipeline_call, total_steps):
        """Run pipeline in a background thread; yield log strings then the PIL Image."""
        step_q     = queue.Queue()
        result_box = [None]
        error_box  = [None]

        def _cb(_, step, _ts, kwargs):
            step_q.put(step + 1)
            return kwargs

        def _run():
            with torch.inference_mode():
                try:
                    result_box[0] = pipeline_call(callback_on_step_end=_cb)
                except Exception as exc:
                    error_box[0] = exc

        t = threading.Thread(target=_run, daemon=True)
        t.start()

        while t.is_alive():
            try:
                step = step_q.get(timeout=0.3)
                yield log_update_last(_progress_bar(step, total_steps))
            except queue.Empty:
                pass

        t.join()
        if error_box[0]:
            raise error_box[0]
        yield result_box[0]  # final item is the PIL Image, not a string

    # Working parameter values
    g_val    = guidance_scale
    s_val    = int(num_steps)
    cn_s_val = cn_scale
    cn_e_val = cn_end
    clow_val = int(canny_low)
    chigh_val= int(canny_high)

    # ── Step 1: Prompt ────────────────────────────────────────────────────────
    if prompt_override.strip():
        prompt = prompt_override.strip()
        yield emit(None, log('✓ Using manual prompt override'))

    elif settings_mode == 'Wild' and story.strip():
        yield emit(None, log('⧗ Wild — Gemini building prompt and tuning for satire...'))
        try:
            prompt, wild = build_wild_settings(story, trigger_word)

            if wild:
                g_val    = float(wild.get('guidance',   g_val))
                s_val    = int(wild.get('steps',        s_val))
                cn_s_val = float(wild.get('cn_scale',   cn_s_val))
                cn_e_val = float(wild.get('cn_end',     cn_e_val))
                clow_val = int(wild.get('canny_low',    clow_val))
                chigh_val= int(wild.get('canny_high',   chigh_val))
                rationale= wild.get('rationale', '')

                mode_map  = {'reimagine': 'Reimagine', 'scene': 'Scene', 'portrait': 'Portrait'}
                suggested = wild.get('mode', '').lower()
                mode_note = ''
                if suggested and suggested != mode:
                    mode_note = f'\n⚡ Wild suggests {mode_map.get(suggested, suggested)} for this story (currently {mode_label})'

                yield emit(
                    None,
                    log(f'✓ Wild settings applied\n   {rationale}{mode_note}'),
                    g=g_val, s=s_val,
                    cn_s=cn_s_val, cn_e=cn_e_val,
                    clow=clow_val, chigh=chigh_val,
                )
            else:
                yield emit(None, log('✓ Prompt ready (settings parse failed — using current sliders)'))

        except Exception as exc:
            yield emit(None, log(f'⚠️  Wild mode failed ({exc}) — falling back to Default'))
            try:
                prompt = build_prompt_from_story(story, trigger_word)
            except Exception:
                prompt = DEFAULT_PROMPT

    elif story.strip():
        yield emit(None, log('⧗ Gemini building your prompt...'))
        try:
            prompt = build_prompt_from_story(story, trigger_word)
            yield emit(None, log('✓ Prompt ready'))
        except Exception as exc:
            yield emit(None, log(f'⚠️  Gemini failed ({exc}) — using default prompt'))
            prompt = DEFAULT_PROMPT

    else:
        prompt = DEFAULT_PROMPT
        yield emit(None, log('✓ No story — using default prompt'))

    trigger = trigger_word.strip()
    if trigger and not prompt.strip().startswith(trigger):
        full_prompt = f'{trigger} {prompt}'.strip()
    else:
        full_prompt = prompt.strip()

    # ── Step 2: Pipeline switch ───────────────────────────────────────────────
    if mode != active_mode:
        yield emit(None, log(f'⧗ Switching to {mode_label} pipeline (~60–90 s)...'))
        load_pipeline(mode)
        yield emit(None, log(f'✓ {mode_label} pipeline ready'))

    # ── Step 3: Resize ────────────────────────────────────────────────────────
    src = Image.fromarray(image).convert('RGB')
    src = src.resize((1024, 1024), Image.LANCZOS)
    generator = torch.Generator(device='cuda').manual_seed(int(seed))

    # ── Step 4: Inference — spinner on, result panel hidden ──────────────────
    result = None

    if mode == 'reimagine':
        yield emit(None, log('⧗ FLUX Kontext rendering...'),
                   spinner=SPINNER_HTML, show_result=False)
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                image=src,
                prompt=full_prompt,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(None, item)   # spinner=None → stays visible unchanged
            else:
                result = item

    elif mode == 'scene':
        yield emit(None, log('⧗ Reading scene depth...'))
        depth_out  = depth_estimator(src)
        depth_arr  = np.array(depth_out['depth'])
        d_min, d_max = depth_arr.min(), depth_arr.max()
        if d_max > d_min:
            depth_norm = ((depth_arr - d_min) / (d_max - d_min) * 255).astype(np.uint8)
        else:
            depth_norm = np.zeros_like(depth_arr, dtype=np.uint8)
        control_image = Image.fromarray(depth_norm).convert('RGB')
        yield emit(None, log('✓ Depth map ready'))
        yield emit(None, log('⧗ FLUX rendering...'),
                   spinner=SPINNER_HTML, show_result=False)
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                control_image=control_image,
                control_mode=2,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                controlnet_conditioning_scale=cn_s_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(None, item)
            else:
                result = item

    elif mode == 'portrait':
        yield emit(None, log('⧗ Extracting portrait outlines...'))
        canny_image = extract_canny(src, clow_val, chigh_val)
        yield emit(None, log('✓ Outlines extracted'))
        yield emit(None, log('⧗ FLUX rendering...'),
                   spinner=SPINNER_HTML, show_result=False)
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                control_image=canny_image,
                control_mode=0,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                controlnet_conditioning_scale=cn_s_val,
                control_guidance_end=cn_e_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(None, item)
            else:
                result = item

    if result is None:
        raise gr.Error('Generation produced no image — check the pipeline logs above.')

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    suffix = {'reimagine': 'kontext', 'scene': 'depth', 'portrait': 'canny'}[mode]
    result.save(f'{OUTPUT_DIR}/{ts}_cartoonify_{suffix}.png')

    gc.collect()
    torch.cuda.empty_cache()

    # spinner='' clears the HTML; image_val != None makes result panel visible again
    yield emit(result, log('✓ Done — saved to Drive'), spinner='')


print('✓ Inference functions ready')


## 10 — Launch Interface

Re-run this cell to restart the UI without reloading models.

**Settings modes:**
- **Default** — uses the hardcoded parameters from `DEFAULTS` for the selected rendering style
- **Wild** — Gemini reads the story and suggests parameters optimised for maximum satirical impact; sliders update to show what's being used

In [ ]:
MODE_DESCRIPTIONS = {
    'Reimagine': 'FLUX reads the full image and recomposes it in cartoon style. Best for creative reinterpretation — scene and staging can change freely.',
    'Scene':     'Depth map preserves the spatial layout and figure positions. Best for crowds, architecture, and compositions where structure matters.',
    'Portrait':  'Canny edges preserve face outlines and silhouettes. Best when a specific person must be immediately recognisable.',
}


def update_mode(mode):
    d = DEFAULTS[mode.lower()]
    is_kontext  = mode == 'Reimagine'
    is_portrait = mode == 'Portrait'
    return (
        MODE_DESCRIPTIONS[mode],
        gr.update(value=d['guidance']),
        gr.update(visible=not is_kontext, value=d['cn_scale']),
        gr.update(visible=is_portrait,    value=d['cn_end']),
        gr.update(visible=is_portrait,    value=d['canny_low']),
        gr.update(visible=is_portrait,    value=d['canny_high']),
    )


CSS = '''
.gradio-container {
    font-family: "Inter", -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: #f0eff5 !important;
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding: 1.5rem !important;
}
#app-header {
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 55%, #24243e 100%);
    border-radius: 18px;
    padding: 2.5rem 2rem 2rem;
    text-align: center;
    margin-bottom: 1.5rem;
    box-shadow: 0 8px 30px rgba(48, 43, 99, 0.35);
}
.app-title {
    font-size: 3rem; font-weight: 900; letter-spacing: -2px;
    color: #ffffff; margin: 0 0 0.5rem; line-height: 1; display: block;
}
.app-subtitle {
    font-size: 1rem; color: #c4b5fd; margin: 0;
    font-weight: 400; line-height: 1.6; display: block;
}
.input-card, .output-card {
    background: #ffffff; border-radius: 16px;
    padding: 1.5rem 1.5rem 1.75rem;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07); border: 1px solid #e5e7eb;
}
.section-label {
    display: block; font-size: 0.68rem; font-weight: 800;
    text-transform: uppercase; letter-spacing: 0.12em;
    color: #7c3aed; margin-bottom: 0.5rem; margin-top: 1.1rem;
}
.section-label:first-child { margin-top: 0; }
.upload-zone {
    border: 2.5px dashed #7c3aed !important; border-radius: 14px !important;
    background: #faf5ff !important; transition: border-color 0.2s, background 0.2s !important;
}
.upload-zone:hover { border-color: #5b21b6 !important; background: #f5edff !important; }
#mode-selector .wrap { display: flex !important; gap: 0.5rem !important; }
#mode-selector label {
    flex: 1 !important; text-align: center !important;
    padding: 0.65rem 0.5rem !important; border: 2px solid #ddd6fe !important;
    border-radius: 10px !important; cursor: pointer !important;
    font-weight: 700 !important; font-size: 0.85rem !important;
    background: #faf5ff !important; color: #5b21b6 !important;
    transition: all 0.2s !important;
}
#mode-selector label:has(input:checked) {
    background: #7c3aed !important; border-color: #7c3aed !important;
    color: white !important;
}
#mode-selector input[type="radio"] { display: none !important; }
.mode-desc p {
    background: #f5f3ff; border: 1px solid #ddd6fe; border-radius: 10px;
    padding: 0.65rem 1rem; font-size: 0.82rem; color: #5b21b6;
    line-height: 1.5; margin: 0.4rem 0 0.4rem;
}
#settings-mode-selector .wrap { display: flex !important; gap: 0.5rem !important; }
#settings-mode-selector label {
    flex: 1 !important; text-align: center !important;
    padding: 0.5rem 0.5rem !important; border: 2px solid #e5e7eb !important;
    border-radius: 10px !important; cursor: pointer !important;
    font-weight: 700 !important; font-size: 0.82rem !important;
    background: #f9fafb !important; color: #6b7280 !important;
    transition: all 0.2s !important;
}
#settings-mode-selector label:has(input:checked) {
    background: #1a1a2e !important; border-color: #1a1a2e !important;
    color: #fbbf24 !important;
}
#settings-mode-selector input[type="radio"] { display: none !important; }
.process-log textarea {
    font-family: "JetBrains Mono", "Fira Code", monospace !important;
    font-size: 0.8rem !important; background: #f9fafb !important;
    border-radius: 10px !important; color: #374151 !important;
}
.output-image { border-radius: 12px !important; overflow: hidden !important; }
#generate-btn {
    background: linear-gradient(135deg, #7c3aed 0%, #5b21b6 100%) !important;
    color: white !important; border: none !important; border-radius: 12px !important;
    font-size: 1.1rem !important; font-weight: 700 !important;
    height: 56px !important; width: 100% !important;
    box-shadow: 0 4px 18px rgba(124,58,237,0.38) !important;
    transition: opacity 0.2s, transform 0.15s !important; margin-top: 1rem !important;
}
#generate-btn:hover  { opacity: 0.90 !important; transform: translateY(-1px) !important; }
#generate-btn:active { transform: translateY(0) !important; }
#generate-btn:disabled { opacity: 0.5 !important; cursor: wait !important; }
footer { display: none !important; }
'''


with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='Cartoonify') as demo:

    gr.HTML('''
        <div id="app-header">
            <span class="app-title">&#127912; Cartoonify</span>
            <span class="app-subtitle">
                Voice up. Turn your story into a visual, satirical illustration.
            </span>
        </div>
    ''')

    with gr.Row(equal_height=False):

        # ── Left column — Inputs ──────────────────────────────────────────────
        with gr.Column(scale=5, min_width=360, elem_classes=['input-card']):

            gr.HTML('<span class="section-label">&#9312; What\'s the story?</span>')
            story_input = gr.Textbox(
                label='',
                placeholder=(
                    'A politician hands out empty promises while people queue for food...\n'
                    'A general behind a desk of medals while tiny soldiers march across a map below...'
                ),
                lines=5,
                max_lines=10,
            )

            gr.HTML('<span class="section-label">&#9313; Upload a photo</span>')
            img_input = gr.Image(
                label='',
                type='numpy',
                elem_classes=['upload-zone'],
                sources=['upload', 'clipboard'],
                height=220,
            )

            gr.HTML('<span class="section-label">&#9314; Rendering style</span>')
            mode_selector = gr.Radio(
                choices=['Reimagine', 'Scene', 'Portrait'],
                value=DEFAULT_MODE,
                label='',
                elem_id='mode-selector',
            )
            mode_desc = gr.Markdown(
                value=MODE_DESCRIPTIONS[DEFAULT_MODE],
                elem_classes=['mode-desc'],
            )

            gr.HTML('<span class="section-label">&#9315; Settings</span>')
            settings_mode_selector = gr.Radio(
                choices=['Default', 'Wild'],
                value=DEFAULT_SETTINGS_MODE,
                label='',
                elem_id='settings-mode-selector',
                info='Default: hardcoded parameters per style  ·  Wild: Gemini tunes parameters for maximum satirical impact',
            )

            with gr.Accordion('⚙️ Advanced Settings', open=False):
                guidance_slider = gr.Slider(
                    minimum=1.0, maximum=10.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['guidance'],
                    step=0.5, label='Guidance Scale',
                    info='Wild mode updates this automatically.',
                )
                steps_slider = gr.Slider(
                    minimum=10, maximum=50, value=DEFAULT_STEPS,
                    step=1, label='Inference Steps',
                    info='Wild mode updates this automatically.',
                )
                cn_scale_slider = gr.Slider(
                    minimum=0.1, maximum=1.5,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_scale'],
                    step=0.05, label='ControlNet Scale',
                    visible=DEFAULT_MODE != 'Reimagine',
                    info='Wild mode updates this automatically.',
                )
                cn_end_slider = gr.Slider(
                    minimum=0.3, maximum=1.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_end'],
                    step=0.05, label='ControlNet Guidance End',
                    visible=DEFAULT_MODE == 'Portrait',
                    info='Wild mode updates this automatically.',
                )
                canny_low_slider = gr.Slider(
                    minimum=0, maximum=200,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_low'],
                    step=10, label='Canny Low Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                    info='Wild mode updates this automatically.',
                )
                canny_high_slider = gr.Slider(
                    minimum=50, maximum=500,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_high'],
                    step=10, label='Canny High Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                    info='Wild mode updates this automatically.',
                )
                seed_input = gr.Number(value=DEFAULT_SEED, label='Seed', precision=0)
                trigger_input = gr.Textbox(
                    value=DEFAULT_TRIGGER, label='LoRA Trigger Word', lines=1,
                )
                with gr.Accordion('Edit prompt directly', open=False):
                    prompt_input = gr.Textbox(
                        label='Prompt override (leave empty to use Gemini + story)',
                        placeholder='Leave empty — Gemini builds the prompt from your story.',
                        lines=3, max_lines=6, value='',
                    )

            generate_btn = gr.Button(
                '\U0001f3a8  Cartoonify  →',
                variant='primary',
                elem_id='generate-btn',
            )

        # ── Right column — Outputs ───────────────────────────────────────────
        with gr.Column(scale=6, min_width=480, elem_classes=['output-card']):

            gr.HTML('<span class="section-label">Processing</span>')
            log_output = gr.Textbox(
                label='',
                lines=5,
                max_lines=8,
                interactive=False,
                elem_classes=['process-log'],
                placeholder='Click Cartoonify to begin...',
                show_label=False,
            )

            gr.HTML('<span class="section-label">&#127917; Result</span>')
            # Animated CSS spinner — shown while FLUX renders, hidden otherwise
            spinner_output = gr.HTML(value='')
            # Result image — hidden during FLUX, shown when result is ready
            result_output = gr.Image(
                label='',
                type='pil',
                elem_classes=['output-image'],
                show_download_button=True,
                height=580,
                interactive=False,
            )

    # ── Event wiring ─────────────────────────────────────────────────────
    mode_selector.change(
        fn=update_mode,
        inputs=[mode_selector],
        outputs=[
            mode_desc,
            guidance_slider,
            cn_scale_slider,
            cn_end_slider,
            canny_low_slider,
            canny_high_slider,
        ],
    )

    generate_btn.click(
        fn=cartoonify,
        inputs=[
            story_input,
            img_input,
            mode_selector,
            settings_mode_selector,
            trigger_input,
            guidance_slider,
            steps_slider,
            cn_scale_slider,
            cn_end_slider,
            canny_low_slider,
            canny_high_slider,
            seed_input,
            prompt_input,
        ],
        outputs=[
            result_output,      # gr.update(visible=False) during FLUX, gr.update(value=img, visible=True) when done
            log_output,
            spinner_output,     # SPINNER_HTML during FLUX, '' when done
            guidance_slider,
            steps_slider,
            cn_scale_slider,
            cn_end_slider,
            canny_low_slider,
            canny_high_slider,
        ],
        api_name='cartoonify',
    )


demo.launch(share=True, show_error=True, quiet=False)
